# 02 - Make the forged dataset (Kaggle)

Session 2 of the plan (P4-P7). **Steps 0-6 are the same setup as notebook 01**, because every new Kaggle session
starts empty (no packages, no TextCtrl code, no Stable Diffusion files). Then:
**7** look at the data and pick pages (P4) - **8** find words with OCR and plan the edits (P5) - **9** load TextCtrl once -
**10** timed pilot on 10 crops (P6) - **11** full run (P6) - **12** paste back, masks, results.csv (P7) - **13** zip the run.

### Before you run anything (Kaggle Settings panel)
1. **Accelerator:** GPU T4 x2. 2. **Internet:** On. 3. **Add Input:** `PATH_TO_DATA500_DATASET` and `weights`.

### How long it takes
About 70-80 minutes in total. Most of it is Step 11 (about 90 crops x ~31 s = ~47 min). You can work on something
else meanwhile. Either keep this tab's session alive, or use **Save Version > Save & Run All** so it runs in the
background and keeps the output. Finished crops are never redone, so a re-run continues where it stopped.

## Step 0 - CONFIG
Every notebook starts with one settings cell, so all the numbers that shape this dataset live in ONE place: the seed,
how many pages to scan (`N_PAGES`), how many crops to pilot (`N_PILOT`), how many forgeries we want (`N_TARGET`),
TextCtrl's settings and the OCR filters. The cell also makes a new run folder `runs/run_YYYYMMDD_HHMM/` and saves the
settings in it as `config.json`, so every result can be traced back to the exact settings that made it.
**Run this cell only ONCE per session** (running it again makes a new, empty run folder). To continue an old run
after a restart, type its folder name into `RESUME_RUN`.

In [ ]:
# import random so we can fix Python's own random numbers
import random
# import numpy so we can fix its random numbers too
import numpy as np
# import torch so we can fix its random numbers too
import torch
# import os so we can make the run folder
import os
# import json so we can save the settings as config.json
import json
# import datetime so the run folder gets the date and time in its name
import datetime

# ---------- reproducibility ----------
# the seed: the same seed gives the same "random" results every run
SEED = 42

# ---------- where things are ----------
# the folder where Kaggle puts the datasets we attached (read-only)
INPUT_ROOT = "/kaggle/input"
# the folder where Kaggle mounts OUR datasets (it includes the Kaggle username)
DATASETS_ROOT = INPUT_ROOT + "/datasets/PATH_TO_KAGGLE_USERNAME"
# the PATH_TO_DATA500_DATASET dataset (500 sample pages)
DATA_ROOT = DATASETS_ROOT + "/PATH_TO_DATA500_DATASET"
# the weights dataset (the 7 .pth files; they sit in an inner weights/ folder)
WEIGHTS_ROOT = DATASETS_ROOT + "/weights"
# the folder we are allowed to write to
WORK_DIR = "/kaggle/working"
# where the TextCtrl code gets cloned
TEXTCTRL_DIR = WORK_DIR + "/TextCtrl"
# the folder TextCtrl reads its weights from (7 .pth files + sd/)
WEIGHTS_DIR = TEXTCTRL_DIR + "/weights"
# the folder for the Stable Diffusion v1.5 parts
SD_DIR = WEIGHTS_DIR + "/sd"
# the GitHub address of TextCtrl
TEXTCTRL_URL = "https://github.com/weichaozeng/TextCtrl.git"
# the Hugging Face address of Stable Diffusion v1.5
SD_REPO = "stable-diffusion-v1-5/stable-diffusion-v1-5"

# ---------- how big the dataset is ----------
# how many randomly chosen pages we may scan with OCR
N_PAGES = 150
# how many crops go into the timed pilot
N_PILOT = 10
# how many forged pages we want (user decision: 100)
N_TARGET = 100

# ---------- TextCtrl settings (the repo's defaults) ----------
# diffusion steps per crop
NUM_STEPS = 50
# the layer where TextCtrl starts mixing in the source style
STARTING_LAYER = 10
# how strongly the target text is pushed into the picture
GUIDANCE_SCALE = 2

# ---------- which OCR words we accept ----------
# 3 to 8 letters or digits, nothing else (TextCtrl cannot draw punctuation or spaces)
WORD_REGEX = r"^[A-Za-z0-9]{3,8}$"
# EasyOCR must be at least 70% sure of the word
MIN_CONFIDENCE = 0.7
# the word's box must be at least 12 pixels tall
MIN_BOX_HEIGHT = 12
# extra pixels kept around each word when we crop it
CROP_PADDING = 3

# ---------- masks and saving ----------
# a grey-level change bigger than this counts as "changed" in the diff mask
DIFF_THRESHOLD = 25
# JPEG quality for the copies in tampered_jpg/
JPEG_QUALITY = 90

# ---------- safety ----------
# the full run refuses to start if the pilot predicts more minutes than this
MAX_FULL_RUN_MINUTES = 75

# ---------- run folder ----------
# leave as None for a new run; to continue an old run, put its name here, e.g. "run_20261024_1530"
RESUME_RUN = None
# if we are continuing an old run, use its name
if RESUME_RUN is not None:
    # reuse the old folder name
    RUN_NAME = RESUME_RUN
# otherwise make a new name from today's date and time
else:
    # e.g. run_20261024_1530
    RUN_NAME = "run_" + datetime.datetime.now().strftime("%Y%m%d_%H%M")
# the full path of the run folder
RUN_DIR = WORK_DIR + "/runs/" + RUN_NAME
# the original word crops
CROPS_SRC = RUN_DIR + "/crops_src"
# TextCtrl's edited crops
CROPS_EDIT = RUN_DIR + "/crops_edit"
# full pages with the edit pasted in (PNG)
TAMPERED_DIR = RUN_DIR + "/tampered"
# the same pages as JPEG quality 90
TAMPERED_JPG_DIR = RUN_DIR + "/tampered_jpg"
# untouched copies of the same pages
ORIGINALS_DIR = RUN_DIR + "/originals"
# answer-key masks: the whole pasted rectangle
MASK_BOX_DIR = RUN_DIR + "/masks/box"
# answer-key masks: only pixels that really changed
MASK_DIFF_DIR = RUN_DIR + "/masks/diff"
# every figure, as PNG for the slides
FIGURES_DIR = RUN_DIR + "/figures"
# make every sub-folder (nothing happens if it already exists)
for folder in [CROPS_SRC, CROPS_EDIT, TAMPERED_DIR, TAMPERED_JPG_DIR, ORIGINALS_DIR, MASK_BOX_DIR, MASK_DIFF_DIR, FIGURES_DIR]:
    # make this folder
    os.makedirs(folder, exist_ok=True)

# fix Python's random generator
random.seed(SEED)
# fix numpy's random generator
np.random.seed(SEED)
# fix torch's random generator
torch.manual_seed(SEED)

# collect the settings that shape the dataset
config = {
    "seed": SEED,
    "n_pages": N_PAGES,
    "n_pilot": N_PILOT,
    "n_target": N_TARGET,
    "num_inference_steps": NUM_STEPS,
    "starting_layer": STARTING_LAYER,
    "guidance_scale": GUIDANCE_SCALE,
    "word_regex": WORD_REGEX,
    "min_confidence": MIN_CONFIDENCE,
    "min_box_height": MIN_BOX_HEIGHT,
    "crop_padding": CROP_PADDING,
    "diff_threshold": DIFF_THRESHOLD,
    "jpeg_quality": JPEG_QUALITY,
    "data_root": DATA_ROOT,
    "run_name": RUN_NAME,
}
# save the settings as config.json inside the run folder
with open(RUN_DIR + "/config.json", "w") as config_file:
    # write them with nice indentation
    json.dump(config, config_file, indent=2)

# print where the run folder is
print("CONFIG loaded. Run folder:", RUN_DIR)
# print the settings
print(json.dumps(config, indent=2))

## Step 1 - Find the datasets, check internet, check the GPU
Before installing anything we check three things. (1) Kaggle really attached our two datasets, and we find where
they are (the folder name can differ slightly from the dataset name). (2) Internet is on, because
downloads (git, pip, Hugging Face) need it. (3) We have a GPU.

**What is a GPU?** Think of the normal chip (the CPU) as one very smart accountant who does one calculation at a time.
A GPU is a stadium of thousands of simple clerks who each do a tiny sum at the same moment. TextCtrl repaints a
picture with millions of small multiplications, repeated 50 times, so a GPU does in seconds what a CPU does in minutes.
TextCtrl needs about 5-6 GB of GPU memory, and a T4 has about 15 GB, so it fits.

In [ ]:
# import sys so we can read the Python version
import sys
# import os so we can look inside folders
import os
# import subprocess so we can run the terminal command nvidia-smi from Python
import subprocess
# import urllib.request so we can test the internet connection
import urllib.request
# import torch, the library that does the GPU maths
import torch


# define a helper that looks for a folder under root (at most max_depth levels down)
def find_folder(root, max_depth, file_name=None, subfolder_suffix=None):
    # walk through every folder under root
    for folder, subfolders, files in os.walk(root):
        # work out how many levels below root this folder is
        depth = folder[len(root):].count(os.sep)
        # if we are looking for a file and this folder contains it, this is the folder
        if file_name is not None and file_name in files:
            # give back this folder
            return folder
        # if we are looking for a subfolder ending in a suffix, check the subfolders
        if subfolder_suffix is not None:
            # look at each subfolder name
            for sub in subfolders:
                # if the name ends with the suffix, this is the folder
                if sub.endswith(subfolder_suffix):
                    # give back this folder
                    return folder
        # do not walk deeper than max_depth (clearing the list stops os.walk from going down)
        if depth >= max_depth:
            # clear the list of subfolders
            subfolders[:] = []
    # nothing found
    return None


# print a header for the dataset check
print("=== Kaggle input datasets ===")
# check that the Kaggle input folder exists at all
if not os.path.isdir(INPUT_ROOT):
    # explain the problem
    print(f"{INPUT_ROOT} not found - are you running on Kaggle?")
    # there is nothing to find
    DATA_DIR = None
    # there is nothing to find
    PTH_DIR = None
# otherwise the input folder exists
else:
    # show the names directly inside /kaggle/input
    print("Inside /kaggle/input:", sorted(os.listdir(INPUT_ROOT))[:20])
    # check that the folder holding our datasets exists
    if os.path.isdir(DATASETS_ROOT):
        # show the names inside it (max 20)
        print("Inside", DATASETS_ROOT, ":", sorted(os.listdir(DATASETS_ROOT))[:20])
    # otherwise that folder does not exist
    else:
        # say so, because the Kaggle username in CONFIG may be wrong
        print("NOT FOUND:", DATASETS_ROOT, "- is the Kaggle username in CONFIG right?")
    # find the folder that holds the bank folders (names ending in _epdf), looking up to 3 levels inside PATH_TO_DATA500_DATASET
    DATA_DIR = find_folder(DATA_ROOT, 3, subfolder_suffix="_epdf")
    # if the pages are in a nested folder (e.g. an inner PATH_TO_DATA500_DATASET/), use it but say so
    if DATA_DIR is not None and DATA_DIR != DATA_ROOT:
        # tell the user we are using the inner folder
        print(f"  NOTE: pages are in a nested folder, using {DATA_DIR} (tell Claude)")
    # find the folder that holds model.pth, looking up to 2 levels inside the weights dataset
    PTH_DIR = find_folder(WEIGHTS_ROOT, 2, file_name="model.pth")
    # if the .pth files are in a nested folder, use it but say so
    if PTH_DIR is not None and PTH_DIR != WEIGHTS_ROOT:
        # tell the user we are using the inner folder
        print(f"  NOTE: .pth files are in a nested folder, using {PTH_DIR} (tell Claude)")

# if we found the pages folder
if DATA_DIR is not None:
    # start counting PNG pages
    n_pages = 0
    # walk through every folder inside the pages folder
    for folder, subfolders, files in os.walk(DATA_DIR):
        # look at each file name
        for file_name in files:
            # count it if it is a PNG
            if file_name.lower().endswith(".png"):
                # add one to the counter
                n_pages += 1
    # print where the pages are and how many there are (we expect 500)
    print(f"  found    pages folder: {DATA_DIR}  ({n_pages} PNG pages, expected 500)")
# otherwise the pages were not found
else:
    # tell the user what to check
    print(f"  MISSING  pages folder (looked inside {DATA_ROOT} for folders ending in _epdf). Did you Add Input > PATH_TO_DATA500_DATASET?")

# if we found the weights folder
if PTH_DIR is not None:
    # count the .pth files directly inside it
    n_pth = len([n for n in os.listdir(PTH_DIR) if n.endswith(".pth")])
    # print where the weights are and how many .pth files there are (we expect 7)
    print(f"  found    weights folder: {PTH_DIR}  ({n_pth} .pth files, expected 7)")
# otherwise the weights were not found
else:
    # tell the user what to check
    print(f"  MISSING  model.pth inside {WEIGHTS_ROOT}. Is the upload finished, and did you Add Input > weights?")

# print an empty line for spacing
print()
# print a header for the internet check
print("=== Internet ===")
# try to open a web page
try:
    # open Hugging Face's home page, giving up after 8 seconds
    urllib.request.urlopen("https://huggingface.co", timeout=8)
    # it worked
    print("Internet: ON")
# if opening failed for any reason
except Exception:
    # explain how to fix it
    print("Internet: OFF - Settings panel > Internet > On (needs phone verification), then restart the session.")

# print an empty line for spacing
print()
# print a header for the software check
print("=== Software ===")
# print the Python version (first word only, e.g. 3.12.x)
print("Python version :", sys.version.split()[0])
# print the torch version
print("torch version  :", torch.__version__)
# ask torch whether it can see a CUDA GPU (True / False)
cuda_ok = torch.cuda.is_available()
# print the answer
print("CUDA available :", cuda_ok)

# if a GPU is available, print its details
if cuda_ok:
    # get the name of GPU number 0
    gpu_name = torch.cuda.get_device_name(0)
    # get the properties (including total memory) of GPU number 0
    props = torch.cuda.get_device_properties(0)
    # convert the memory from bytes to gigabytes (1 GB = 1024**3 bytes)
    total_gb = props.total_memory / 1024**3
    # print the GPU name
    print("GPU name       :", gpu_name)
    # print the GPU memory with one decimal
    print(f"GPU memory     : {total_gb:.1f} GB")
    # check whether it looks like the T4 we expect
    if "T4" not in gpu_name:
        # warn that this is a different GPU (not a problem, just good to know)
        print("Note: this is not a T4, but it should still work.")
# otherwise there is no GPU
else:
    # explain how to fix it
    print("NO GPU FOUND. Go to Runtime > Change runtime type > T4 GPU, then run this cell again.")

# print an empty line for spacing
print()
# print a header for the nvidia-smi output
print("=== nvidia-smi (the GPU's own status report) ===")
# try to run nvidia-smi
try:
    # run the command and capture what it prints as text
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    # print what nvidia-smi said
    print(result.stdout)
# if the command does not exist, there is no GPU driver
except FileNotFoundError:
    # tell the user nvidia-smi is not available
    print("nvidia-smi not found (this means no GPU runtime is selected).")

## Step 2 - Get TextCtrl's code
So far we only checked the machine (Step 1). Now we download the TextCtrl program itself from GitHub, like
downloading a recipe book. The recipes are the Python files; the ingredients (the trained weights) come later.
Kaggle starts every session with a clean `/kaggle/working`, so this cell clones only if the folder is not already there.

In [ ]:
# import os so we can check and list folders
import os
# import subprocess so we can run git from Python
import subprocess

# check whether the repo is already cloned (a .git folder exists inside it)
if os.path.isdir(TEXTCTRL_DIR + "/.git"):
    # say we are skipping the download
    print("TextCtrl is already cloned, skipping.")
# otherwise we need to clone it
else:
    # clone only the newest snapshot (--depth 1 means no history, so it is fast)
    result = subprocess.run(["git", "clone", "--depth", "1", TEXTCTRL_URL, TEXTCTRL_DIR], capture_output=True, text=True)
    # print the end of git's messages
    print(result.stdout[-500:], result.stderr[-500:])
    # if git failed (code is not 0), stop with a clear message
    if result.returncode != 0:
        # raise an error so the notebook stops here
        raise RuntimeError("git clone failed - read the message above")

# list what is inside the cloned folder, in alphabetical order
names = sorted(os.listdir(TEXTCTRL_DIR))
# print how many items there are
print(f"{len(names)} items in {TEXTCTRL_DIR}")
# print the first 20 names only
print(names[:20])
# ask git which exact version (commit) we got
commit = subprocess.run(["git", "-C", TEXTCTRL_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
# print the commit so we can reproduce this exact code later
print("commit:", commit)

## Step 3 - Install only what is missing
TextCtrl ships a shopping list (`requirement.txt`) written for Python 3.8 and torch 1.13 (2022-2023). Kaggle today
have a much newer Python and torch. Buying everything on the old list would overwrite things Kaggle already has and break
them. So we do it like checking the fridge first: for each package, if Python can already find it we skip it,
otherwise we install the **newest** version (no old pin). **torch is never touched.**
Each package prints one plain-English line saying what it is for. If one fails we stop and fix that one first.

Notebook 02 also installs **EasyOCR**, the tool that finds words (and their boxes) on each page.

In [ ]:
# import sys so we know which Python is running
import sys
# import subprocess so we can run pip from Python
import subprocess
# import importlib.util so we can ask "is this package installed?"
import importlib.util
# import importlib so we can refresh Python's package list after installing
import importlib

# make an empty list that will collect (package, result) pairs for the summary
results = []

# define a helper that installs one package only if Python cannot find it
def install_if_missing(pip_name, import_name, eli5):
    # tell the user what this package is, in one simple line
    print(f"- {pip_name}: {eli5}")
    # ask Python whether it can already find the package
    already_there = importlib.util.find_spec(import_name) is not None
    # if it is already installed, we do nothing
    if already_there:
        # report that we skipped it
        print("    already installed, skipped")
        # remember the result for the summary
        results.append((pip_name, "already installed"))
        # stop here for this package
        return
    # otherwise run pip to install it, capturing the messages pip prints
    run = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], capture_output=True, text=True)
    # if pip finished with code 0 it worked
    if run.returncode == 0:
        # report success
        print("    installed OK")
        # remember the result for the summary
        results.append((pip_name, "installed now"))
    # otherwise pip failed
    else:
        # say it failed
        print("    FAILED. Last lines of pip's message:")
        # show only the last 15 lines of the error to keep the output short
        print("\n".join(run.stderr.strip().split("\n")[-15:]))
        # remember the failure for the summary
        results.append((pip_name, "FAILED"))

# download tool for Hugging Face (needed in Step 4)
install_if_missing("huggingface_hub", "huggingface_hub", "lets Python download files from the Hugging Face website")
# safe file format for model weights
install_if_missing("safetensors", "safetensors", "a safe, fast file format for saved model weights (the Stable Diffusion files use it)")
# library of ready-made models
install_if_missing("transformers", "transformers", "a big shelf of ready-made text and vision models (TextCtrl's requirement list includes it)")
# helper for loading big models
install_if_missing("accelerate", "accelerate", "helps load big models onto the GPU smoothly (in TextCtrl's requirement list)")
# readable tensor reshaping
install_if_missing("einops", "einops", "reshapes big grids of numbers (tensors) with short, readable commands")
# settings-file reader
install_if_missing("omegaconf", "omegaconf", "reads settings files (.yaml) and turns them into easy-to-use objects")
# torch wrapper
install_if_missing("pytorch-lightning", "pytorch_lightning", "a tidy wrapper around torch; TextCtrl's inference.py uses its seed_everything function")
# perceptual image distance
install_if_missing("lpips", "lpips", "a ruler that measures how different two images LOOK to a neural network")
# image-model shelf
install_if_missing("timm", "timm", "a shelf of ready-made image models (the text-reading ViTSTR model is built from it)")
# natural file sorting
install_if_missing("natsort", "natsort", "sorts file names the human way: img2 before img10")

# word finder for Step 8
install_if_missing("easyocr", "easyocr", "reads the text in a picture and tells us where each word is (the find-a-word step)")

# make Python notice the packages we just installed
importlib.invalidate_caches()
# print an empty line for spacing
print()
# print a title for the summary table
print("=== Summary ===")
# loop over the collected results
for pip_name, status in results:
    # print one row per package
    print(f"  {pip_name:18} {status}")

### Step 3b - Does TextCtrl's own code import?
Installing packages is like buying ingredients; this cell checks the recipe actually starts. We try to `import` the
exact modules that `inference.py` uses. We try ALL of them and show the end of each error (several may share one cause). The usual fix is one more missing
package or one old-code incompatibility, and we fix those one at a time (and log each in `notes/decisions.md`).
If this cell prints an error, copy the last ~30 lines to Claude.

**Fix found here (error 1):** TextCtrl ships its OWN old copy of `diffusers`, and it asks `huggingface_hub` for two
tools (`HfFolder`, `cached_download`) that new versions removed. The small "shim" block below adds those two names
back at runtime, so we do not have to edit TextCtrl's files. Any later notebook that imports TextCtrl needs the same block.

**Fix found here (error 2):** the old `diffusers` also loads its JAX/Flax parts when `jax` and `flax` are installed (Kaggle and Colab have them), and new JAX removed `jax.random.KeyArray`. TextCtrl runs on torch and never uses JAX, so the cell sets the old library's own switch `USE_FLAX=0`. It also makes Python forget any half-loaded copy of `diffusers` left over from an earlier failed run (otherwise the new setting is ignored until you restart the runtime).

**Fix found here (error 3):** the old `diffusers` asks `transformers` for `CLIPFeatureExtractor`, which was renamed to `CLIPImageProcessor`. The cell adds the old name back as another name for the new class.

**Fix found here (error 4):** in a FRESH Kaggle session the first import pass failed on `CLIPFeatureExtractor` even though
the shim had run, while a second pass in the same session worked. The cell now tries the whole import test up to two
times, re-applying the CLIP name to whatever `transformers` module Python is using at that moment, and prints whether
that module was swapped. A failed attempt 1 followed by `All imports OK` on attempt 2 is fine.

In [ ]:
# import sys so we can tell Python where TextCtrl's code is
import sys
# import os so we can move into the repo folder
import os
# import importlib so we can import modules by name
import importlib
# import traceback so we can print the error details if an import fails
import traceback

# let Python find TextCtrl's modules (src/ and utils.py)
sys.path.insert(0, TEXTCTRL_DIR)
# move into the repo folder, because its scripts use relative paths like weights/model.pth
os.chdir(TEXTCTRL_DIR)

# switch off the Flax/JAX parts of TextCtrl's old diffusers (its own official switch; TextCtrl uses torch, and new JAX broke those parts)
os.environ["USE_FLAX"] = "0"
# loop over a copy of the names of every module Python has already loaded
for loaded_name in list(sys.modules):
    # pick out modules that belong to TextCtrl's bundled diffusers or its src package
    if loaded_name == "diffusers" or loaded_name.startswith("diffusers.") or loaded_name.startswith("src"):
        # forget them, so they are re-imported with the new setting instead of reusing a half-loaded copy from an earlier failed run
        del sys.modules[loaded_name]

# import huggingface_hub so we can add two old names that TextCtrl's bundled diffusers still asks for
import huggingface_hub

# define a stand-in for HfFolder, an old helper that new huggingface_hub versions removed
class HfFolderShim:
    # the old code only calls HfFolder.get_token(), so that is the only thing we provide
    @staticmethod
    # define get_token
    def get_token():
        # return the saved Hugging Face login token (or None if we are not logged in)
        return huggingface_hub.get_token()

# only add the stand-in if the real HfFolder is gone
if not hasattr(huggingface_hub, "HfFolder"):
    # add our stand-in under the old name
    huggingface_hub.HfFolder = HfFolderShim
# only add this if the old cached_download is gone
if not hasattr(huggingface_hub, "cached_download"):
    # the old cached_download is replaced by hf_hub_download in new versions
    huggingface_hub.cached_download = huggingface_hub.hf_hub_download

# define a stand-in for Repository, an old helper for pushing models to Hugging Face (we never push)
class RepositoryShim:
    # if anything ever tries to create one, stop with a clear message
    def __init__(self, *args, **kwargs):
        # explain why this is not available
        raise RuntimeError("Repository (pushing to Hugging Face) is not available and not needed for inference")

# only add the stand-in if the real Repository is gone
if not hasattr(huggingface_hub, "Repository"):
    # add our stand-in under the old name
    huggingface_hub.Repository = RepositoryShim

# import the place where the old code looks for three error classes
import huggingface_hub.utils as hub_utils
# import the place where new versions keep those error classes
import huggingface_hub.errors as hub_errors
# loop over the three error class names the old code imports
for error_name in ["EntryNotFoundError", "RepositoryNotFoundError", "RevisionNotFoundError"]:
    # if the old location no longer has this name
    if not hasattr(hub_utils, error_name):
        # copy it over from the new location
        setattr(hub_utils, error_name, getattr(hub_errors, error_name))

# import transformers so we can add one more old name that TextCtrl's bundled diffusers asks for
import transformers
# always store the old name CLIPFeatureExtractor directly on the module (it was renamed to CLIPImageProcessor; same job)
transformers.CLIPFeatureExtractor = transformers.CLIPImageProcessor

# print a title for the shim report (if you do NOT see this line, you are running an OLD copy of this cell)
print("=== Step 3b shim report (cell version 5) ===")
# print the transformers version
print("transformers version      :", transformers.__version__)
# print the huggingface_hub version
print("huggingface_hub version   :", huggingface_hub.__version__)
# print whether HfFolder is now available
print("HfFolder available        :", hasattr(huggingface_hub, "HfFolder"))
# print whether Repository is now available
print("Repository available      :", hasattr(huggingface_hub, "Repository"))
# print whether CLIPFeatureExtractor is now available
print("CLIPFeatureExtractor      :", hasattr(transformers, "CLIPFeatureExtractor"))
# print the Flax switch
print("USE_FLAX                  :", os.environ.get("USE_FLAX"))
# print an empty line for spacing
print()

# the modules that inference.py imports, in the order it imports them
modules_to_test = ["pytorch_lightning", "src.trainer.CtrlBase", "src.MuSA.GaMuSA_app", "src.MuSA.GaMuSA", "utils"]
# remember whether everything worked
all_ok = False
# run the whole import test up to two times (in a fresh session the first pass can fail, see the markdown above)
for attempt in [1, 2]:
    # get the transformers module Python is using RIGHT NOW (it may have been swapped during an earlier import)
    current_transformers = sys.modules["transformers"]
    # store the old CLIP name on it again
    current_transformers.CLIPFeatureExtractor = current_transformers.CLIPImageProcessor
    # loop over a copy of the names of every loaded module
    for loaded_name in list(sys.modules):
        # pick out TextCtrl's bundled diffusers and its src package
        if loaded_name == "diffusers" or loaded_name.startswith("diffusers.") or loaded_name.startswith("src"):
            # forget them, so this attempt imports them cleanly
            del sys.modules[loaded_name]
    # assume this attempt works until an import fails
    attempt_ok = True
    # print which attempt this is
    print(f"--- import attempt {attempt} ---")
    # try each module one by one
    for module_name in modules_to_test:
        # try to import it
        try:
            # do the import
            importlib.import_module(module_name)
            # report success
            print(f"  OK      {module_name}")
        # if anything goes wrong, handle it
        except Exception:
            # report the failure
            print(f"  FAILED  {module_name}")
            # show only the last 8 lines of the error
            print("\n".join(traceback.format_exc().strip().split("\n")[-8:]))
            # remember that this attempt failed
            attempt_ok = False
    # report whether the transformers module was swapped for a different object (our diagnosis)
    print("transformers module swapped during import:", sys.modules["transformers"] is not transformers)
    # if every import worked, we are done
    if attempt_ok:
        # remember success
        all_ok = True
        # stop trying
        break

# print the final verdict
print("All imports OK" if all_ok else "Some imports failed - send the messages above to Claude")

## Step 4 - Stable Diffusion v1.5 weights (download only what is missing)
TextCtrl is built on top of Stable Diffusion, the general-purpose "painter": `vae/` zips and unzips images, `unet/`
is the painter's hand, `scheduler/` is the step-by-step recipe. They are not in your Kaggle datasets, so we download
them from Hugging Face into `TextCtrl/weights/sd/` (inside `/kaggle/working`, because `/kaggle/input` is read-only).
The repo stores each model in several formats (`.bin`, `.safetensors`, `fp16`, `non_ema`); we take the **`.bin`** file per folder (TextCtrl's config names the `.bin` files, and the old `diffusers` copy bundled with it cannot read `.safetensors`)
plus its small config, about 3.8 GB instead of ~20 GB. The cell checks every file first and skips what is already there.

In [ ]:
# import os so we can check files and make folders
import os
# import time so we can time the download
import time
# import the Hugging Face download function
from huggingface_hub import snapshot_download

# the files we need from vae/ (settings + the .bin weights file that TextCtrl asks for)
vae_files = ["vae/config.json", "vae/diffusion_pytorch_model.bin"]
# the files we need from unet/ (settings + the big .bin weights file that TextCtrl asks for)
unet_files = ["unet/config.json", "unet/diffusion_pytorch_model.bin"]
# the file we need from scheduler/ (just the settings)
scheduler_files = ["scheduler/scheduler_config.json"]
# put all wanted files in one list
wanted = vae_files + unet_files + scheduler_files

# make sure the sd/ folder exists
os.makedirs(SD_DIR, exist_ok=True)
# start an empty list for files that are not here yet
missing = []
# check every wanted file
for rel_path in wanted:
    # build the full path of this file
    full_path = os.path.join(SD_DIR, rel_path)
    # a file counts as present only if it exists and is not empty
    present = os.path.isfile(full_path) and os.path.getsize(full_path) > 0
    # if it is not present, add it to the missing list
    if not present:
        # remember it
        missing.append(rel_path)

# print how many files we need and how many are missing
print(f"Need {len(wanted)} files, missing {len(missing)}")
# if nothing is missing we can skip the download
if len(missing) == 0:
    # say so
    print("Everything is already here - no download.")
# otherwise download the missing files
else:
    # print which files will be downloaded
    print("Downloading:", missing)
    # note the start time
    start = time.time()
    # download only the missing files from Hugging Face straight into the sd/ folder
    snapshot_download(repo_id=SD_REPO, local_dir=SD_DIR, allow_patterns=missing)
    # work out how long it took
    seconds = time.time() - start
    # print the time
    print(f"Download finished in {seconds:.0f} seconds")

# print an empty line for spacing
print()
# print a title for the file check
print("=== What is in weights/sd/ now ===")
# check every wanted file again
for rel_path in wanted:
    # build the full path
    full_path = os.path.join(SD_DIR, rel_path)
    # if the file exists, print its size in megabytes
    if os.path.isfile(full_path):
        # convert bytes to megabytes (1 MB = 1024 * 1024 bytes)
        size_mb = os.path.getsize(full_path) / (1024 * 1024)
        # print one row
        print(f"  OK       {rel_path:50} {size_mb:8.1f} MB")
    # otherwise it is still missing
    else:
        # print a warning row
        print(f"  MISSING  {rel_path}")

## Step 5 - Put the 7 .pth files next to sd/ and check everything
TextCtrl looks for its files at `TextCtrl/weights/...` (the 7 `.pth` files directly inside, `sd/` as a subfolder).
Our `.pth` files live in the read-only `weights` dataset, so we make a **symbolic link** (a desktop-style shortcut)
for each one inside `TextCtrl/weights/`. Nothing is copied, so no extra disk space is used. Then we list the files so
you can tick off the 7 `.pth` files and the Stable Diffusion parts with your own eyes.

In [ ]:
# import os so we can make links and list files
import os

# the 7 weight files TextCtrl's README says must be directly inside weights/
expected_pth = ["model.pth", "text_encoder.pth", "style_encoder.pth", "vision_model.pth", "ocr_model.pth", "vgg19.pth", "vitstr_base_patch16_224.pth"]

# stop early if Step 1 did not find the weights dataset
if PTH_DIR is None:
    # stop with a clear message
    raise RuntimeError("PTH_DIR is None: the weights dataset was not found in Step 1. Check the upload and Add Input.")

# make sure the weights folder exists (Step 4 already created it together with sd/)
os.makedirs(WEIGHTS_DIR, exist_ok=True)
# make one shortcut per .pth file
for name in expected_pth:
    # where the real file lives (read-only dataset)
    real_file = os.path.join(PTH_DIR, name)
    # where TextCtrl will look for it
    link_file = os.path.join(WEIGHTS_DIR, name)
    # if the real file is missing, say so and move on
    if not os.path.isfile(real_file):
        # print a warning
        print(f"  not in dataset: {name}")
        # go to the next file
        continue
    # if an old shortcut is there, remove only the shortcut (no data is deleted)
    if os.path.islink(link_file):
        # remove the old shortcut
        os.remove(link_file)
    # if nothing is there now, make the shortcut
    if not os.path.exists(link_file):
        # create the shortcut
        os.symlink(real_file, link_file)

# print an empty line for spacing
print()
# print a title for the .pth check
print("=== The 7 .pth files (seen from TextCtrl/weights/) ===")
# count how many we find
found_count = 0
# check each expected file
for name in expected_pth:
    # build its path inside TextCtrl/weights/
    file_path = os.path.join(WEIGHTS_DIR, name)
    # if the file can be opened through the shortcut, print its size
    if os.path.isfile(file_path):
        # convert bytes to megabytes
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        # print one row
        print(f"  OK       {name:32} {size_mb:9.1f} MB")
        # count it
        found_count += 1
    # otherwise it is missing
    else:
        # print a warning row
        print(f"  MISSING  {name}")

# print an empty line for spacing
print()
# print a title for the sd check
print("=== Stable Diffusion parts (seen from TextCtrl/weights/) ===")
# count the SD files we find
sd_found = 0
# check each wanted SD file (the list from Step 4)
for rel_path in wanted:
    # build its path inside TextCtrl/weights/sd/
    file_path = os.path.join(SD_DIR, rel_path)
    # if it exists, count it and print it
    if os.path.isfile(file_path):
        # count it
        sd_found += 1
        # print one row
        print(f"  OK       sd/{rel_path}")
    # otherwise it is missing
    else:
        # print a warning row
        print(f"  MISSING  sd/{rel_path}")

# print an empty line for spacing
print()
# list the first 20 names directly inside weights/ to spot extra or nested folders
print("Everything directly inside TextCtrl/weights/ (max 20):", sorted(os.listdir(WEIGHTS_DIR))[:20])
# print the final verdict
print(f"\nVERDICT: {found_count}/7 .pth files, {sd_found}/{len(wanted)} Stable Diffusion files")

## Step 6 - Four fixes found by reading TextCtrl's code BEFORE running it
Reading `inference.py`, `utils.py` and the config ahead of time showed four things that would break on today's
Python 3.13 / torch 2.11. Fixing them now saves four failed runs. None of them edits a TextCtrl file.

1. **`torch.load` is stricter now.** Since torch 2.6 it refuses to open older checkpoint files unless told it is
   safe (`weights_only=False`). TextCtrl calls it in 6 places. We set the old default for this notebook session. This
   is safe here because the only files opened are the authors' `.pth` files and Stable Diffusion's own weights, not files from strangers.
2. **`np.int` no longer exists.** One TextCtrl file still writes `np.int`, which numpy deleted. We add the old names
   back as plain aliases (`np.int = int`).
3. **A font file is missing.** The config names `weights/arial.ttf`, which is not in your dataset. It is probably
   only used for training-log pictures, but as cheap insurance we link a look-alike font (Liberation Sans is made to
   match Arial's letter sizes).
4. **Pre-flight check.** We list every file the config names and confirm each one exists, so a missing file shows up
   here as one clear line instead of as a long error later.

**Expected output:** three short "done" lines, then a list of `OK` file checks ending with `PRE-FLIGHT: READY`.
If you restart the Kaggle session, run Step 3b again too, because these fixes live only in the running session.

In [ ]:
# import os so we can check files and make links
import os
# import numpy so we can add back its old names
import numpy as np
# import torch so we can change how torch.load behaves
import torch

# ---------- fix 1: torch.load keeps its OLD default (weights_only=False) ----------
# only wrap torch.load once, even if this cell is run twice
if not getattr(torch.load, "textctrl_patched", False):
    # remember the real torch.load
    original_torch_load = torch.load
    # define our version, which accepts the same inputs
    def torch_load_old_default(*args, **kwargs):
        # if the caller did not choose weights_only, choose False (the old behaviour)
        kwargs.setdefault("weights_only", False)
        # call the real torch.load with the settings
        return original_torch_load(*args, **kwargs)
    # leave a label so we know it is already wrapped
    torch_load_old_default.textctrl_patched = True
    # replace torch.load with our version
    torch.load = torch_load_old_default
# report that fix 1 is done
print("fix 1 done: torch.load uses weights_only=False (trusted files only)")

# ---------- fix 2: give numpy back its removed names ----------
# go through each removed name and the plain Python type it used to mean
for alias_name, python_type in [("int", int), ("float", float), ("bool", bool), ("object", object)]:
    # only add the alias if numpy does not have it
    if not hasattr(np, alias_name):
        # add the old name back
        setattr(np, alias_name, python_type)
# report that fix 2 is done
print("fix 2 done: np.int / np.float / np.bool / np.object aliases available")

# ---------- fix 3: a stand-in for weights/arial.ttf ----------
# where TextCtrl's config expects the font
font_link = WEIGHTS_DIR + "/arial.ttf"
# if a font is already there, leave it alone
if os.path.exists(font_link):
    # report it
    print("fix 3 done: weights/arial.ttf already exists")
# otherwise look for a look-alike font on this machine
else:
    # the fonts we would accept, best first
    candidates = ["/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf", "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"]
    # nothing chosen yet
    chosen_font = None
    # check each candidate
    for candidate in candidates:
        # take the first one that exists
        if os.path.isfile(candidate) and chosen_font is None:
            # remember it
            chosen_font = candidate
    # if we found one, link it under the name TextCtrl expects
    if chosen_font is not None:
        # make the shortcut
        os.symlink(chosen_font, font_link)
        # report which font stands in
        print("fix 3 done: weights/arial.ttf now points to", chosen_font)
    # otherwise there is no stand-in
    else:
        # say so; it only matters for training-log pictures
        print("fix 3 skipped: no look-alike font found (probably only needed for training-log pictures)")

# ---------- fix 4: pre-flight check of every file the config names ----------
# print an empty line for spacing
print()
# print a title for the check
print("=== Pre-flight check ===")
# the files inference.py and configs/inference.yaml need
needed_files = [
    TEXTCTRL_DIR + "/configs/inference.yaml",
    TEXTCTRL_DIR + "/example/i_s.txt",
    TEXTCTRL_DIR + "/example/i_t.txt",
    WEIGHTS_DIR + "/model.pth",
    WEIGHTS_DIR + "/text_encoder.pth",
    WEIGHTS_DIR + "/ocr_model.pth",
    WEIGHTS_DIR + "/vision_model.pth",
    WEIGHTS_DIR + "/sd/vae/config.json",
    WEIGHTS_DIR + "/sd/vae/diffusion_pytorch_model.bin",
    WEIGHTS_DIR + "/sd/unet/diffusion_pytorch_model.bin",
    WEIGHTS_DIR + "/sd/scheduler/scheduler_config.json",
]
# assume everything is fine until a file is missing
all_present = True
# check each file
for needed in needed_files:
    # if it exists, print OK
    if os.path.isfile(needed):
        # print the short name
        print("  OK       " + needed.replace(TEXTCTRL_DIR + "/", ""))
    # otherwise print MISSING
    else:
        # print the short name
        print("  MISSING  " + needed.replace(TEXTCTRL_DIR + "/", ""))
        # remember that something is missing
        all_present = False
# print the verdict
print("PRE-FLIGHT: READY" if all_present else "PRE-FLIGHT: NOT READY - if the .bin files are missing, re-run Step 4 first")

## Step 7 - Look at the data and pick candidate pages (P4)
Before planning any forgery we look at what is really inside `PATH_TO_DATA500_DATASET` instead of assuming: which file types,
how many bank folders, how many pages, and how big a page is. Then we pick `N_PAGES` = 150 pages at random with the
seed (so the same 150 come out every time). We pick more pages than we need because some pages will have no
suitable word. On Kaggle the dataset sits on a fast local disk, so unlike the old Colab plan we read pages
directly and do not copy them first.

**Expected output:** file types (about 500 `.png` plus `sample_list.txt`), ~18 bank folders, 500 pages, five page
sizes around 2480 pixels wide, `Picked 150 candidate pages`, and a figure of 4 pages saved as `figures/sample_pages.png`.
**Write down:** how many pages there are and the typical page size.

In [ ]:
# import random so we can pick pages with the seed
import random
# import Path so paths with spaces are handled safely
from pathlib import Path
# import Counter so we can count file types
from collections import Counter
# import PIL so we can read image sizes and open pages
from PIL import Image
# import matplotlib so we can show pages
import matplotlib.pyplot as plt

# stop early if Step 1 did not find the pages
if DATA_DIR is None:
    # stop with a clear message
    raise RuntimeError("DATA_DIR is None - run Step 1 and check the PATH_TO_DATA500_DATASET dataset is attached")
# the pages folder as a Path object
data_root = Path(DATA_DIR)

# count every file by its type (extension)
type_counts = Counter()
# walk through every file anywhere in the dataset
for file_path in data_root.rglob("*"):
    # only count files, not folders
    if file_path.is_file():
        # add one to the count of this file type
        type_counts[file_path.suffix.lower()] += 1
# print a title
print("=== What is inside PATH_TO_DATA500_DATASET ===")
# print the file types and how many of each
print("File types:", dict(type_counts))

# list the bank folders at the top level
bank_folders = sorted([p.name for p in data_root.iterdir() if p.is_dir()])
# print how many bank folders there are (and up to 20 names)
print(f"Bank folders: {len(bank_folders)} -> {bank_folders[:20]}")

# list every page, as a path relative to the dataset folder, in a fixed (sorted) order
all_pages = sorted([p.relative_to(data_root) for p in data_root.rglob("*.png")])
# print how many pages there are
print("PNG pages:", len(all_pages))
# print one example path so we see the folder layout
print("Example page path:", all_pages[0])

# make a random generator that starts from our seed
rng = random.Random(SEED)
# pick N_PAGES different pages at random
candidate_pages = rng.sample(all_pages, N_PAGES)

# print a title for the sizes
print("Sizes of 5 sampled pages (width x height, pixels):")
# look at the first 5 sampled pages
for rel_path in candidate_pages[:5]:
    # read the size from the image header
    page_width, page_height = Image.open(data_root / rel_path).size
    # print it
    print(f"  {page_width} x {page_height}   {rel_path}")

# save the list of sampled pages in the run folder
with open(RUN_DIR + "/candidate_pages.txt", "w") as list_file:
    # write one page per line
    for rel_path in candidate_pages:
        # write the relative path
        list_file.write(str(rel_path) + "\n")
# report what we picked
print(f"Picked {len(candidate_pages)} candidate pages with seed {SEED} (list saved to candidate_pages.txt)")

# make a figure with 4 pages stacked on top of each other (pages are wide and short)
fig, axes = plt.subplots(4, 1, figsize=(10, 12))
# fill one row per page
for i, rel_path in enumerate(candidate_pages[:4]):
    # open the page
    page_img = Image.open(data_root / rel_path).convert("RGB")
    # draw it
    axes[i].imshow(page_img)
    # label it with its bank folder and size
    axes[i].set_title(f"{rel_path.parts[0]}  -  {page_img.size[0]} x {page_img.size[1]} px", fontsize=9)
    # hide the pixel-position numbers
    axes[i].axis("off")
# give the figure a title
fig.suptitle("4 of the sampled bank-statement pages (top part of each statement)", fontsize=11)
# tidy the spacing
fig.tight_layout()
# save the figure for the slides
fig.savefig(FIGURES_DIR + "/sample_pages.png", dpi=120)
# show it
plt.show()

## Step 8 - Find words with OCR and plan one edit per page (P5)
Now we choose WHAT to forge. EasyOCR reads each page and returns every word with a box and a confidence score.
We keep a word only if it is 3-8 letters/digits (TextCtrl cannot draw punctuation or spaces), EasyOCR is at least 70%
sure, and the box is at least 12 px tall. From the survivors we pick ONE per page, preferring **numbers** (changing
figures is the realistic bank-statement fraud), then all-lowercase words, then words with capitals.

**How the new text is made:** numbers change 1-2 digits but keep their length (e.g. `4521` -> `4821`); words become a
different bank word of the same length (`total` -> `funds`), keeping the capital-letter pattern; anything else
changes one letter. Each word is cropped with 3 px padding and saved with a SAFE name (`crop_0001.png`), because
page names contain spaces and TextCtrl's text files split on spaces. `plan.csv` records which crop came from which
page and box. We stop as soon as `N_TARGET` pages have a plan.

**Expected output:** OCR progress every 10 pages, then a summary: pages scanned, pages with a usable word, the
"pages scanned per usable word" ratio (a finding for the slides), counts of numbers vs words and of words with
capitals, and a figure of 8 crops with their planned edit (`figures/planned_edits.png`). EasyOCR downloads its own
small model the first time (needs Internet On).

In [ ]:
# import os so we can check files
import os
# import re so we can test words against the regex
import re
# import time so we can time the OCR
import time
# import random so the choices follow the seed
import random
# import Path so paths with spaces are handled safely
from pathlib import Path
# import numpy so we can hand pages to EasyOCR
import numpy as np
# import pandas so we can save plan.csv
import pandas as pd
# import torch so we can free GPU memory afterwards
import torch
# import matplotlib so we can show crops
import matplotlib.pyplot as plt
# import PIL so we can open and crop pages
from PIL import Image

# where the plan is saved
plan_path = RUN_DIR + "/plan.csv"

# small lists of bank / invoice words, grouped by length (used to replace words of the same length)
WORD_BANK = {
    3: ["fee", "tax", "due", "net", "pay", "atm"],
    4: ["cash", "bank", "loan", "paid", "fund", "bill", "rate", "debt"],
    5: ["total", "funds", "asset", "debit", "order", "payee"],
    6: ["amount", "credit", "refund", "salary", "charge", "branch"],
    7: ["balance", "deposit", "account", "payment", "pending", "savings"],
    8: ["transfer", "interest", "merchant", "purchase", "dividend", "customer"],
}


# define a helper that copies the capital-letter pattern of the old word onto the new word
def match_case(new_word, old_word):
    # ALL CAPS stays all caps
    if old_word.isupper():
        # return the new word in capitals
        return new_word.upper()
    # a capital first letter stays a capital first letter
    if old_word[0].isupper():
        # return the new word with a capital first letter
        return new_word.capitalize()
    # otherwise use lower case
    return new_word.lower()


# define a helper that changes 1 or 2 digits of a number but keeps its length
def change_digits(text, rng):
    # turn the text into a list of single digits
    digits = list(text)
    # decide whether to change 1 or 2 digits
    how_many = rng.choice([1, 2])
    # pick which positions to change
    positions = rng.sample(range(len(digits)), how_many)
    # change each chosen position
    for pos in positions:
        # every digit except the current one
        options = [d for d in "0123456789" if d != digits[pos]]
        # do not create a new leading zero
        if pos == 0 and digits[0] != "0":
            # remove 0 from the options
            options = [d for d in options if d != "0"]
        # pick the new digit
        digits[pos] = rng.choice(options)
    # glue the digits back together
    return "".join(digits)


# define a helper that changes exactly one letter (keeping its case)
def change_one_letter(text, rng):
    # find the positions that hold letters
    letter_positions = [i for i, c in enumerate(text) if c.isalpha()]
    # pick one of them
    pos = rng.choice(letter_positions)
    # remember the old letter
    old_letter = text[pos]
    # every letter except the old one
    options = [c for c in "abcdefghijklmnopqrstuvwxyz" if c != old_letter.lower()]
    # pick the new letter
    new_letter = rng.choice(options)
    # keep it a capital if the old one was a capital
    if old_letter.isupper():
        # make it a capital
        new_letter = new_letter.upper()
    # rebuild the word with the new letter
    return text[:pos] + new_letter + text[pos + 1:]


# define a helper that makes the target text for one source word
def make_target(text, rng):
    # numbers: change 1-2 digits
    if text.isdigit():
        # return the changed number
        return change_digits(text, rng)
    # pure words: try a different bank word of the same length
    if text.isalpha():
        # bank words of the same length that differ from the source word
        options = [w for w in WORD_BANK.get(len(text), []) if w != text.lower()]
        # if there is at least one, use it
        if len(options) > 0:
            # pick one and copy the capital-letter pattern
            return match_case(rng.choice(options), text)
    # otherwise change one letter
    return change_one_letter(text, rng)


# if a plan already exists (we are continuing an old run), load it instead of redoing the OCR
if os.path.exists(plan_path):
    # read it, keeping the texts as text (so 0123 does not turn into 123)
    plan = pd.read_csv(plan_path, dtype={"source_text": str, "target_text": str})
    # report it
    print(f"plan.csv already exists ({len(plan)} rows) - skipping OCR (continuing an old run)")
# otherwise run the OCR and make the plan
else:
    # import EasyOCR here (installed in Step 3)
    import easyocr
    # load EasyOCR for English, on the GPU
    reader = easyocr.Reader(["en"], gpu=True)
    # a random generator that starts from our seed
    rng = random.Random(SEED)
    # prepare the word filter
    word_pattern = re.compile(WORD_REGEX)
    # the list of planned edits
    rows = []
    # count pages we read
    pages_scanned = 0
    # count pages that had at least one usable word
    pages_with_word = 0
    # start the stopwatch
    start = time.time()
    # go through the candidate pages in their random order
    for rel_path in candidate_pages:
        # stop as soon as we have enough plans
        if len(rows) >= N_TARGET:
            # leave the loop
            break
        # count this page
        pages_scanned += 1
        # open the page
        page_img = Image.open(Path(DATA_DIR) / rel_path).convert("RGB")
        # remember its size
        page_width, page_height = page_img.size
        # ask EasyOCR for every word: a box (4 corner points), the text and a confidence
        detections = reader.readtext(np.array(page_img))
        # the usable words on this page
        candidates = []
        # check each detected word
        for bbox, text, confidence in detections:
            # the x positions of the 4 corners
            xs = [point[0] for point in bbox]
            # the y positions of the 4 corners
            ys = [point[1] for point in bbox]
            # turn the corners into a straight rectangle: left and right
            x1, x2 = int(min(xs)), int(max(xs))
            # top and bottom
            y1, y2 = int(min(ys)), int(max(ys))
            # rule 1: only 3-8 letters/digits
            if not word_pattern.match(text):
                # skip this word
                continue
            # rule 2: EasyOCR must be sure enough
            if confidence < MIN_CONFIDENCE:
                # skip this word
                continue
            # rule 3: the box must be tall enough
            if (y2 - y1) < MIN_BOX_HEIGHT:
                # skip this word
                continue
            # priority 0 = pure number (best)
            if text.isdigit():
                # numbers first
                priority = 0
            # priority 1 = no capital letters
            elif not any(c.isupper() for c in text):
                # lowercase words second
                priority = 1
            # priority 2 = has capital letters
            else:
                # words with capitals last
                priority = 2
            # keep this word as a candidate
            candidates.append((priority, text, confidence, x1, y1, x2, y2))
        # if nothing on this page passed the rules, go to the next page
        if len(candidates) == 0:
            # next page
            continue
        # count this page as useful
        pages_with_word += 1
        # the best (lowest) priority on this page
        best_priority = min(c[0] for c in candidates)
        # all candidates with that priority
        best_candidates = [c for c in candidates if c[0] == best_priority]
        # pick one of them at random (with the seed)
        priority, text, confidence, x1, y1, x2, y2 = rng.choice(best_candidates)
        # add padding on the left, without going off the page
        x1 = max(0, x1 - CROP_PADDING)
        # add padding on top
        y1 = max(0, y1 - CROP_PADDING)
        # add padding on the right
        x2 = min(page_width, x2 + CROP_PADDING)
        # add padding at the bottom
        y2 = min(page_height, y2 + CROP_PADDING)
        # the number of this plan (1, 2, 3, ...)
        number = len(rows) + 1
        # a safe crop name with no spaces
        crop_name = f"crop_{number:04d}.png"
        # a safe page name used for every output of this page
        page_id = f"page_{number:04d}"
        # cut out the word and save it
        page_img.crop((x1, y1, x2, y2)).save(CROPS_SRC + "/" + crop_name)
        # make the new text
        target_text = make_target(text, rng)
        # remember everything about this planned edit
        rows.append({
            "id": number,
            "crop_name": crop_name,
            "page_id": page_id,
            "page_path": str(rel_path),
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "source_text": text,
            "target_text": target_text,
            "kind": "number" if text.isdigit() else "word",
            "has_upper": any(c.isupper() for c in text),
            "ocr_confidence": round(float(confidence), 3),
        })
        # print progress every 10 pages
        if pages_scanned % 10 == 0:
            # show pages read, plans made and time so far
            print(f"  scanned {pages_scanned} pages, {len(rows)} plans, {time.time() - start:.0f} s")
    # turn the list into a table
    plan = pd.DataFrame(rows)
    # save it as plan.csv
    plan.to_csv(plan_path, index=False)
    # write i_s.txt (what each crop shows now) in TextCtrl's format: "filename text"
    with open(RUN_DIR + "/i_s.txt", "w") as is_file:
        # one line per crop
        for row in rows:
            # file name, one space, text
            is_file.write(f"{row['crop_name']} {row['source_text']}\n")
    # write i_t.txt (what each crop should show) in the same format
    with open(RUN_DIR + "/i_t.txt", "w") as it_file:
        # one line per crop
        for row in rows:
            # file name, one space, text
            it_file.write(f"{row['crop_name']} {row['target_text']}\n")
    # free the GPU memory EasyOCR used, so TextCtrl has room
    del reader
    # ask torch to hand the memory back
    torch.cuda.empty_cache()
    # print a title for the summary
    print()
    # print the summary title
    print("=== OCR + planning summary ===")
    # pages we had to read
    print(f"Pages scanned            : {pages_scanned}")
    # pages that had at least one usable word
    print(f"Pages with a usable word : {pages_with_word}")
    # plans made
    print(f"Plans made               : {len(rows)} (target {N_TARGET})")
    # the ratio is a finding for the slides
    print(f"Pages scanned per plan   : {pages_scanned / max(len(rows), 1):.2f}")
    # time taken
    print(f"OCR time                 : {time.time() - start:.0f} s")
    # warn if we ran out of pages
    if len(rows) < N_TARGET:
        # explain what to change
        print(f"WARNING: only {len(rows)} plans - raise N_PAGES in CONFIG to scan more pages")

# print how many numbers vs words were planned
print("Kind counts:", plan["kind"].value_counts().to_dict())
# print how many words contain capitals
print("Has capitals:", plan["has_upper"].value_counts().to_dict())

# show the first 8 crops with their planned edit, 2 rows of 4
fig, axes = plt.subplots(2, 4, figsize=(12, 3.4))
# fill one box per crop
for i, row in enumerate(plan.head(8).to_dict("records")):
    # find this crop's box in the 2 x 4 grid
    ax = axes[i // 4][i % 4]
    # draw the crop
    ax.imshow(Image.open(CROPS_SRC + "/" + row["crop_name"]).convert("RGB"))
    # label it with the planned edit
    ax.set_title(f"{row['source_text']} -> {row['target_text']}", fontsize=10)
    # hide the pixel-position numbers
    ax.axis("off")
# give the figure a title
fig.suptitle("8 planned edits: the cropped word and its new text", fontsize=11)
# tidy the spacing
fig.tight_layout()
# save the figure for the slides
fig.savefig(FIGURES_DIR + "/planned_edits.png", dpi=150)
# show it
plt.show()

## Step 9 - Load TextCtrl once
In the smoke test, TextCtrl's own `inference.py` loaded the model (~5 GB of weights) every time it ran. Here we load it
ONCE and keep it in GPU memory, so the pilot (Step 10) and the full run (Step 11) can share it. We use TextCtrl's own
functions (`create_model`, `load_state_dict`, `GaMuSA`) with exactly the settings `inference.py` uses, so the results
match what the smoke test produced. Steps 3b and 6 must have run in this session (they hold the compatibility fixes).

**Expected output:** a few loading lines from TextCtrl, then `TextCtrl loaded in ~60-120 s`, GPU memory about 4-5 GB.

In [ ]:
# import os so we can move into the repo folder
import os
# import sys so Python can find TextCtrl's code
import sys
# import time so we can time the loading
import time
# import torch so we can read the GPU memory
import torch

# move into the repo folder (TextCtrl's config uses relative paths like weights/model.pth)
os.chdir(TEXTCTRL_DIR)
# make sure Python can import inference.py
if TEXTCTRL_DIR not in sys.path:
    # add the repo folder
    sys.path.insert(0, TEXTCTRL_DIR)
# import the repo's inference.py, which brings in all the functions we need
import inference

# start the stopwatch
start = time.time()
# build the model from TextCtrl's config file and put it on the GPU
model = inference.create_model("configs/inference.yaml").cuda()
# load the trained weights into it (same call as inference.py)
model.load_state_dict(inference.load_state_dict("weights/model.pth"), strict=False)
# switch to "use" mode (not training)
model.eval()
# the text-checker settings, copied exactly from inference.py
monitor_cfg = {
    "max_length": 25,
    "loss_weight": 1.,
    "attention": "position",
    "backbone": "transformer",
    "backbone_ln": 3,
    "checkpoint": "weights/vision_model.pth",
    "charset_path": "src/module/abinet/data/charset_36.txt",
}
# wrap the model in TextCtrl's editing pipeline
pipeline = inference.GaMuSA(model, monitor_cfg)
# report the loading time and GPU memory
print(f"TextCtrl loaded in {time.time() - start:.0f} s; GPU memory in use: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Step 10 - Timed pilot on the first 10 crops (P6)
Before committing ~50 minutes of GPU time we edit only `N_PILOT` = 10 crops and time each one. Like a test batch in
a kitchen: if the first 10 take far longer than expected, or look terrible, we find out now, not after an hour.
Each crop is edited with TextCtrl's own `text_editing` function and saved to `crops_edit/` under the same safe name.
Every crop is seeded with the same seed before editing, so a crop gives the same result even if the run is resumed.

**Expected output:** one line per crop with its edit and seconds (~30 s each on a T4), the median seconds per crop,
the estimated minutes for the remaining ~90 crops (about 45-50 min), `OK to start the full run`, and a figure of 6
before/after pairs (`figures/pilot_pairs.png`). **Write down:** pilot seconds per crop and your gut reaction to the 6 pairs.

In [ ]:
# import os so we can check files
import os
# import time so we can time each crop
import time
# import random so we can seed each crop
import random
# import numpy for seeding and image conversion
import numpy as np
# import torch for seeding
import torch
# import matplotlib so we can show before/after pairs
import matplotlib.pyplot as plt
# import PIL so we can open and save crops
from PIL import Image

# turn the plan table into a list of dictionaries (one per crop)
plan_rows = plan.to_dict("records")


# define a helper that edits one crop with TextCtrl; returns True if it edited, False if it was already done
def edit_one_crop(row):
    # where the original crop is
    source_path = CROPS_SRC + "/" + row["crop_name"]
    # where the edited crop goes
    output_path = CROPS_EDIT + "/" + row["crop_name"]
    # if it was already edited earlier, skip it (makes the run resumable)
    if os.path.exists(output_path):
        # nothing to do
        return False
    # remember the crop's original size
    width, height = Image.open(source_path).size
    # seed Python's random numbers (same seed for every crop)
    random.seed(SEED)
    # seed numpy's random numbers
    np.random.seed(SEED)
    # seed torch's random numbers
    torch.manual_seed(SEED)
    # seed the GPU's random numbers
    torch.cuda.manual_seed_all(SEED)
    # load the crop as the "source" picture (resized to 256 x 256 inside, like inference.py)
    source_image = inference.load_image(source_path)
    # load the same crop as the "style" picture (inference.py does the same)
    style_image = inference.load_image(source_path)
    # ask TextCtrl to repaint the crop with the target text
    result = inference.text_editing(pipeline, source_image, style_image, str(row["source_text"]), str(row["target_text"]), starting_layer=STARTING_LAYER, ddim_steps=NUM_STEPS, scale=GUIDANCE_SCALE)
    # the second item of the result is the edited picture (numbers between 0 and 1)
    edited_array = result[1]
    # turn it into an image and resize it back to the crop's original size
    edited_img = Image.fromarray((edited_array * 255).astype(np.uint8)).resize((width, height))
    # save the edited crop
    edited_img.save(output_path)
    # report that we edited it
    return True


# the pilot crops
pilot_rows = plan_rows[:N_PILOT]
# the seconds each newly edited pilot crop took
pilot_times = []
# print a title
print(f"=== Pilot: {len(pilot_rows)} crops ===")
# edit each pilot crop
for row in pilot_rows:
    # start the stopwatch for this crop
    start = time.time()
    # edit it (or skip it if already done)
    did_edit = edit_one_crop(row)
    # how long it took
    seconds = time.time() - start
    # only time crops that were really edited now
    if did_edit:
        # remember the time
        pilot_times.append(seconds)
    # print one line per crop
    print(f"  {row['crop_name']}: {row['source_text']} -> {row['target_text']}  ({seconds:.1f} s{'' if did_edit else ', already done'})")

# if we timed at least one crop, estimate the full run
if len(pilot_times) > 0:
    # the median is not thrown off by one slow first crop
    pilot_seconds_per_crop = float(np.median(pilot_times))
    # how many crops are left after the pilot
    remaining = len(plan_rows) - len(pilot_rows)
    # estimated minutes for the rest
    estimate_minutes = pilot_seconds_per_crop * remaining / 60
    # print the median time
    print(f"Median seconds per crop : {pilot_seconds_per_crop:.1f} s")
    # print the estimate
    print(f"Estimate for the other {remaining} crops: {estimate_minutes:.0f} minutes")
    # compare with the safety limit
    if estimate_minutes > MAX_FULL_RUN_MINUTES:
        # tell the user to stop and decide
        print(f"STOP: over {MAX_FULL_RUN_MINUTES} min - ask Claude (cut NUM_STEPS to 25, or N_TARGET)")
    # otherwise it is fine
    else:
        # tell the user to go on
        print("OK to start the full run (Step 11)")
# otherwise the pilot crops were all done in an earlier session
else:
    # no new timing
    pilot_seconds_per_crop = None
    # say so
    print("Pilot crops were already edited earlier - no new timing")

# show 6 before/after pairs from the pilot: one row per crop
shown = pilot_rows[:6]
# 6 rows, 2 columns: before | after
fig, axes = plt.subplots(len(shown), 2, figsize=(6, 1.2 * len(shown)), squeeze=False)
# fill one row per crop
for i, row in enumerate(shown):
    # draw the original crop
    axes[i][0].imshow(Image.open(CROPS_SRC + "/" + row["crop_name"]).convert("RGB"))
    # label it
    axes[i][0].set_title("before: " + str(row["source_text"]), fontsize=9)
    # hide the axis numbers
    axes[i][0].axis("off")
    # draw the edited crop
    axes[i][1].imshow(Image.open(CROPS_EDIT + "/" + row["crop_name"]).convert("RGB"))
    # label it
    axes[i][1].set_title("after: " + str(row["target_text"]), fontsize=9)
    # hide the axis numbers
    axes[i][1].axis("off")
# give the figure a title
fig.suptitle("Pilot: 6 crops before and after TextCtrl", fontsize=11)
# tidy the spacing
fig.tight_layout()
# save the figure for the slides
fig.savefig(FIGURES_DIR + "/pilot_pairs.png", dpi=150)
# show it
plt.show()

## Step 11 - Full run on the remaining crops (P6)
The pilot passed, so now TextCtrl edits every other crop in `plan.csv`. This is the long wait (~45-50 minutes for ~90
crops). It prints progress every 10 crops with an estimate of the minutes left. The cell refuses to start if the
pilot predicted more than `MAX_FULL_RUN_MINUTES`. If the session stops halfway, just run Steps 0-6 and 9 again (with
`RESUME_RUN` set in CONFIG) and re-run this cell: finished crops are skipped.

**Expected output:** progress lines like `30/90 crops, 15.2 min so far, about 30 min left`, then
`crops_edit/ has 100 / 100 edited crops` and `FULL RUN COMPLETE`. **Write down:** the full-run time.

In [ ]:
# import os so we can check files
import os
# import time so we can time the run
import time

# the crops after the pilot
rest_rows = plan_rows[N_PILOT:]
# the pilot's seconds per crop, if the pilot ran in this session
pilot_speed = globals().get("pilot_seconds_per_crop")
# if we know the speed and the estimate is over the limit, stop
if pilot_speed is not None and pilot_speed * len(rest_rows) / 60 > MAX_FULL_RUN_MINUTES:
    # stop with a clear message
    raise RuntimeError(f"Estimated {pilot_speed * len(rest_rows) / 60:.0f} min is over MAX_FULL_RUN_MINUTES - ask Claude first")

# print a title
print(f"=== Full run: {len(rest_rows)} crops ===")
# start the stopwatch
start = time.time()
# count crops edited in this session
done_now = 0
# edit each remaining crop, counting from 1
for index, row in enumerate(rest_rows, start=1):
    # edit it (or skip it if it was already done)
    if edit_one_crop(row):
        # count it
        done_now += 1
    # every 10 crops (and at the end), print progress
    if index % 10 == 0 or index == len(rest_rows):
        # minutes since the start
        elapsed_minutes = (time.time() - start) / 60
        # average minutes per newly edited crop so far
        minutes_per_crop = elapsed_minutes / max(done_now, 1)
        # rough minutes left
        minutes_left = (len(rest_rows) - index) * minutes_per_crop
        # print the progress line
        print(f"  {index}/{len(rest_rows)} crops, {elapsed_minutes:.1f} min so far, about {minutes_left:.0f} min left")

# count how many plan crops now have an edited version
edited_count = sum(1 for row in plan_rows if os.path.exists(CROPS_EDIT + "/" + row["crop_name"]))
# print the count
print(f"crops_edit/ has {edited_count} / {len(plan_rows)} edited crops")
# print the total time
print(f"Full-run time: {(time.time() - start) / 60:.1f} min ({done_now} crops edited in this session)")
# print the verdict
print("FULL RUN COMPLETE" if edited_count == len(plan_rows) else "INCOMPLETE - re-run this cell to continue (finished crops are skipped)")

## Step 12 - Paste back, make the answer-key masks, save results.csv (P7)
Each edited crop goes back into its page at exactly the box it came from, which turns a crop into a forged page.
For every page we save:
- `tampered/page_XXXX.png` - the forged page as **PNG**. PNG is lossless: it stores the pixels exactly, so the only
  traces in the file are our edit. JPEG would blur the picture in 8x8 blocks and add compression traces of its own,
  which a detector might "find" even where we changed nothing.
- `tampered_jpg/page_XXXX.jpg` - a JPEG copy at quality 90, because Week 5 detectors like ADCD-Net may expect JPEG input.
- `originals/page_XXXX.png` - the untouched page, so Week 5 can measure false alarms (a detector flagging a clean page).
- `masks/box/page_XXXX.png` - the **box mask**: the whole pasted rectangle is white. Every pixel in it was replaced,
  so this is the main answer key.
- `masks/diff/page_XXXX.png` - the **diff mask**: only pixels whose grey level changed by more than 25, cleaned with
  a small morphological open (removes lonely specks) and close (fills tiny holes). Stricter, but 25 is our own choice.

Every output of a page shares one safe name (`page_0001`), and `results.csv` links it back to the original page path.

**Expected output:** counts of files in each folder (about 100 each), the average share of the box that really
changed, and a figure of 6 pages zoomed in around the edit: original | tampered | box mask | diff mask
(`figures/paste_and_masks.png`). We zoom in because a whole page is ~2480 px wide and the edit would be invisible.
**Write down:** the final dataset count and the box-vs-diff mask choice (worth a slide line).

In [ ]:
# import os so we can count files
import os
# import shutil so we can copy the original pages byte for byte
import shutil
# import numpy for the mask maths
import numpy as np
# import pandas so we can save results.csv
import pandas as pd
# import cv2 for the morphological clean-up of the diff mask
import cv2
# import Path so page paths with spaces are handled safely
from pathlib import Path
# import PIL so we can open, paste and save images
from PIL import Image
# import matplotlib for the figure
import matplotlib.pyplot as plt

# a small 3 x 3 square used to clean the diff mask
kernel = np.ones((3, 3), np.uint8)
# the rows of results.csv
result_rows = []
# go through every planned edit
for row in plan_rows:
    # the edited crop for this page
    edited_path = CROPS_EDIT + "/" + row["crop_name"]
    # if TextCtrl did not produce it, skip this page
    if not os.path.exists(edited_path):
        # say which one is missing
        print("  skipped (no edited crop):", row["crop_name"])
        # go to the next page
        continue
    # the safe file name shared by every output of this page
    page_name = row["page_id"] + ".png"
    # the box we cropped (left, top, right, bottom)
    x1, y1, x2, y2 = int(row["x1"]), int(row["y1"]), int(row["x2"]), int(row["y2"])
    # the original page inside the dataset
    original_path = Path(DATA_DIR) / row["page_path"]
    # open the original page
    original_img = Image.open(original_path).convert("RGB")
    # open the edited crop and make sure it is exactly the box size
    edited_crop = Image.open(edited_path).convert("RGB").resize((x2 - x1, y2 - y1))
    # start the forged page as a copy of the original
    tampered_img = original_img.copy()
    # paste the edited crop into the box
    tampered_img.paste(edited_crop, (x1, y1))
    # save the forged page as PNG
    tampered_img.save(TAMPERED_DIR + "/" + page_name)
    # save a JPEG copy at quality 90
    tampered_img.save(TAMPERED_JPG_DIR + "/" + row["page_id"] + ".jpg", quality=JPEG_QUALITY)
    # copy the untouched original page byte for byte
    shutil.copyfile(original_path, ORIGINALS_DIR + "/" + page_name)

    # page size (width, height)
    page_width, page_height = original_img.size
    # box mask: start all black
    box_mask = np.zeros((page_height, page_width), np.uint8)
    # make the pasted rectangle white
    box_mask[y1:y2, x1:x2] = 255
    # the original page in grey levels (as signed numbers so we can subtract)
    grey_before = np.array(original_img.convert("L"), dtype=np.int16)
    # the forged page in grey levels
    grey_after = np.array(tampered_img.convert("L"), dtype=np.int16)
    # white where the grey level changed by more than the threshold
    diff_mask = (np.abs(grey_after - grey_before) > DIFF_THRESHOLD).astype(np.uint8) * 255
    # morphological open: remove lonely white specks
    diff_mask = cv2.morphologyEx(diff_mask, cv2.MORPH_OPEN, kernel)
    # morphological close: fill tiny black holes inside changed areas
    diff_mask = cv2.morphologyEx(diff_mask, cv2.MORPH_CLOSE, kernel)
    # save the box mask
    Image.fromarray(box_mask).save(MASK_BOX_DIR + "/" + page_name)
    # save the diff mask
    Image.fromarray(diff_mask).save(MASK_DIFF_DIR + "/" + page_name)

    # number of pixels in the box
    box_pixels = int((box_mask > 0).sum())
    # number of pixels that really changed
    diff_pixels = int((diff_mask > 0).sum())
    # copy the plan row so we can add columns
    result = dict(row)
    # path of the edited crop (relative to the run folder)
    result["crop_edit"] = "crops_edit/" + row["crop_name"]
    # path of the forged PNG page
    result["tampered_png"] = "tampered/" + page_name
    # path of the JPEG copy
    result["tampered_jpg"] = "tampered_jpg/" + row["page_id"] + ".jpg"
    # path of the untouched page
    result["original_png"] = "originals/" + page_name
    # path of the box mask
    result["mask_box"] = "masks/box/" + page_name
    # path of the diff mask
    result["mask_diff"] = "masks/diff/" + page_name
    # pixels in the box
    result["box_pixels"] = box_pixels
    # pixels that changed
    result["diff_pixels"] = diff_pixels
    # share of the box that changed
    result["diff_share_of_box"] = round(diff_pixels / max(box_pixels, 1), 3)
    # keep the row
    result_rows.append(result)

# turn the rows into a table
results = pd.DataFrame(result_rows)
# save it as results.csv
results.to_csv(RUN_DIR + "/results.csv", index=False)

# print a title
print("=== Dataset saved ===")
# count files in each output folder
for folder in [TAMPERED_DIR, TAMPERED_JPG_DIR, ORIGINALS_DIR, MASK_BOX_DIR, MASK_DIFF_DIR]:
    # print the folder (relative to the run folder) and its file count
    print(f"  {folder.replace(RUN_DIR + '/', ''):14} {len(os.listdir(folder))} files")
# print the number of rows in results.csv
print(f"results.csv rows: {len(results)}")
# print the average share of the box that really changed
print(f"Average share of the box that changed (diff / box): {results['diff_share_of_box'].mean():.2f}")

# figure: 6 pages zoomed in around the edit, 4 columns
shown = result_rows[:6]
# how many extra pixels to show around the box
margin = 80
# 6 rows, 4 columns
fig, axes = plt.subplots(len(shown), 4, figsize=(12, 1.6 * len(shown)), squeeze=False)
# the column titles
column_titles = ["original", "tampered", "box mask", "diff mask"]
# fill one row per page
for i, result in enumerate(shown):
    # the zoom window: left
    zx1 = max(0, int(result["x1"]) - margin)
    # top
    zy1 = max(0, int(result["y1"]) - margin)
    # right
    zx2 = int(result["x2"]) + margin
    # bottom
    zy2 = int(result["y2"]) + margin
    # the four pictures to show, in column order
    paths = [result["original_png"], result["tampered_png"], result["mask_box"], result["mask_diff"]]
    # draw each column
    for col, rel_path in enumerate(paths):
        # open the picture and cut out the zoom window (anything past the page edge is filled with black)
        zoom = Image.open(RUN_DIR + "/" + rel_path).crop((zx1, zy1, zx2, zy2))
        # draw it (masks in grey levels)
        axes[i][col].imshow(zoom, cmap="gray")
        # hide the axis numbers
        axes[i][col].axis("off")
        # put the column title on the top row only
        if i == 0:
            # write the title
            axes[i][col].set_title(column_titles[col], fontsize=10)
    # label the row with the edit on the left picture
    axes[i][0].text(0, -5, f"{result['source_text']} -> {result['target_text']}", fontsize=8)
# give the figure a title
fig.suptitle("6 forged pages, zoomed in around the edit", fontsize=11)
# tidy the spacing
fig.tight_layout()
# save the figure for the slides
fig.savefig(FIGURES_DIR + "/paste_and_masks.png", dpi=150)
# show it
plt.show()

## Step 13 - Zip the run folder so you can download it
Everything from this session lives in one run folder. Zipping it makes ONE file to download from Kaggle's Output tab
(instead of hundreds of small files), and it is also what notebook 03 reads. Download only this zip, not the whole
output (the whole output also contains the ~7 GB of Stable Diffusion files).

**Expected output:** `Zip saved: /kaggle/working/run_YYYYMMDD_HHMM.zip` with its size (roughly 50-150 MB) and a
final count of files per folder.

In [ ]:
# import os so we can count files and read the zip size
import os
# import shutil so we can make the zip
import shutil

# make RUN_NAME.zip in /kaggle/working, containing the run folder
zip_path = shutil.make_archive(WORK_DIR + "/" + RUN_NAME, "zip", root_dir=WORK_DIR + "/runs", base_dir=RUN_NAME)
# the zip size in megabytes
zip_mb = os.path.getsize(zip_path) / (1024 * 1024)
# print where it is and how big it is
print(f"Zip saved: {zip_path} ({zip_mb:.0f} MB)")
# print a title for the final count
print("Files per folder in the run:")
# go through every folder inside the run folder
for folder, subfolders, files in sorted(os.walk(RUN_DIR)):
    # print the folder (relative to the run folder) and how many files it holds
    print(f"  {folder.replace(RUN_DIR, '') or '/':16} {len(files)} files")

In [ ]:
import os
from pathlib import Path
from zipfile import ZipFile, ZIP_STORED
from IPython.display import FileLink, display

base = Path("/kaggle/working")
archive = base / "kaggle_working_backup.zip"

# Collect existing files first, excluding the backup itself.
files = [
    path
    for path in base.rglob("*")
    if path.is_file() and path != archive
]

print(f"Backing up {len(files)} files...", flush=True)

# No compression: faster packaging, but a larger download.
with ZipFile(archive, "w", compression=ZIP_STORED,
             allowZip64=True) as z:
    for path in files:
        z.write(
            path,
            arcname=str(path.relative_to(base.parent))
        )

print(f"Backup ready: {archive.stat().st_size / 1024**3:.2f} GiB")

previous_directory = os.getcwd()
try:
    os.chdir(base)
    display(FileLink(
        archive.name,
        result_html_prefix="Download your complete backup: "
    ))
finally:
    os.chdir(previous_directory)